# 01 — Validate the book

Load the book from `data/holding.xlsx` for a given as-of date and check
that every portfolio reconciles cleanly: weights sum to 1.0, every ticker
resolves against `STASHAWAY_UNIVERSE`, and role tags are present.

**Inputs:** `data/holding.xlsx` (`Holdings` + `PortfolioValue` sheets).

**Outputs:** summary table; per-portfolio detail; final assertions.


In [ ]:
from datetime import date
from pathlib import Path

import pandas as pd

from hailmary.allocation.holdings_book import load_book
from hailmary.allocation.portfolios import Role

HOLDING_XLSX = Path('../../data/holding.xlsx')
AS_OF = date(2026, 5, 31)
assert HOLDING_XLSX.exists(), f'Holding sheet not found at {HOLDING_XLSX}'


## Load the book

In [ ]:
portfolios = load_book(HOLDING_XLSX, as_of=AS_OF)
print(f'Loaded {len(portfolios)} portfolios at {AS_OF}')


## Reconcile weights

Each portfolio's weights sum to 1.0 ± 1e-4 — already enforced inside
`Portfolio.__post_init__`. This cell is a final visual check.


In [ ]:
rows = [
    {
        'name': p.name,
        'currency': p.currency,
        'total_value': p.total_value,
        'n_holdings': len(p.holdings),
        'weight_sum': sum(h.weight for h in p.holdings),
    }
    for p in portfolios
]
summary = pd.DataFrame(rows).sort_values('total_value', ascending=False)
summary


## Role tagging

`load_book` applies role tags from `book_config.ROLES`. Portfolios without
an entry default to `{HOLDING}`. New sleeves on the sheet that aren't yet
in `book_config.ROLES` will show here so you know to tag them.


In [ ]:
from hailmary.allocation.book_config import ROLES

untagged = [p.name for p in portfolios if p.name not in ROLES]
if untagged:
    print('Portfolios with no explicit role tag (defaults applied):')
    for n in untagged:
        print(f'  - {n!r}')
else:
    print(f'All {len(portfolios)} portfolios have explicit role tags.')


## Diagnostic vs hidden portfolios

In [ ]:
diag_rows = [
    {
        'name': p.name,
        'roles': ','.join(sorted(r.value for r in p.roles)),
        'currency': p.currency,
        'total_value': p.total_value,
        'n_holdings': len(p.holdings),
    }
    for p in portfolios
]
diag = pd.DataFrame(diag_rows)
in_diag = diag[diag['roles'].str.contains('holding')]
hidden = diag[~diag['roles'].str.contains('holding')]
print(f'In diagnostic ({len(in_diag)}):')
display(in_diag.sort_values('total_value', ascending=False))
print(f'Hidden ({len(hidden)}):')
display(hidden)

## Detailed holdings — first three portfolios

In [ ]:
for p in portfolios[:3]:
    print(f'\n=== {p.name} ({p.currency}) — total {p.total_value:,.2f} ===')
    rows = [
        {
            'ticker': h.ticker,
            'yahoo_ticker': h.metadata.ticker,
            'asset_class': h.metadata.asset_class,
            'region': h.metadata.region,
            'sector': h.metadata.sector,
            'weight': h.weight,
            'value': h.value,
        }
        for h in p.holdings
    ]
    display(pd.DataFrame(rows))

## Final assertion

Fix any issue here before running notebooks 02 / 03 / 04.


In [ ]:
assert len(portfolios) >= 15, f'Expected >= 15 portfolios, got {len(portfolios)}'
for p in portfolios:
    assert abs(sum(h.weight for h in p.holdings) - 1.0) < 1e-4, p.name
managed = [p for p in portfolios if Role.MANAGED_BENCHMARK in p.roles]
assert len(managed) >= 1, 'No MANAGED_BENCHMARK portfolios — benchmark deltas will be empty'
print('All checks passed — proceed to notebook 02 / 03 / 04.')
